In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

In [2]:
TASK = Path.cwd().parent
ROOT = TASK.parent
INPUT, OUTPUT = TASK/'input', TASK/'output'
DIAGNOSTICS = INPUT/'diagnostics'
CONFIG = json.loads((ROOT/'config.json').read_text())
Y, SEED = CONFIG['last_year'], CONFIG['seed']
MODELS = CONFIG['models']
EMPIRICAL = 'empirical_complete'
SOURCES = [EMPIRICAL] + MODELS
COMMON_START = 6
MIN_REENTRY_RISK = 20
LABELS = {'arw4':'ARW4', 'lognormal_grw':'GRW', 'ar1_s':'AR(1)',
          'hurdle_ar1_s_p':'Hurdle AR(1)', 'ar6_6':'AR(6,6)', 'se_hurdle_s':'Self-exciting hurdle'}

In [3]:
counts = pd.read_csv(DIAGNOSTICS/'count_moments.csv')
increments = pd.read_csv(DIAGNOSTICS/'increment_moments.csv')
annual = pd.read_csv(DIAGNOSTICS/'annual_moments.csv')
ranks = pd.read_csv(DIAGNOSTICS/'rank_mixing.csv')
hazards = pd.read_csv(DIAGNOSTICS/'restart_hazard.csv')

In [4]:
# complete empirical cohort for every curve; available-case curves remain upstream
STATISTICS = ['mean', 'median', 'variance', 'sd', 'q05', 'q95']
levels = counts.loc[counts.source.isin(SOURCES)].melt(
    id_vars=['source', 'age', 'scale', 'n'], value_vars=STATISTICS, var_name='statistic', value_name='value')
levels['domain'] = 'annual'
changes = increments.loc[increments.source.isin(SOURCES)].melt(
    id_vars=['source', 'age', 'scale', 'n'], value_vars=STATISTICS, var_name='statistic', value_name='value')
# use destination age for increments; fully generated endpoints start at 6 -> 7
changes['age'] += 1
changes['domain'] = 'increment'
activity = annual.loc[annual.source.isin(SOURCES), ['source', 'age', 'n', 'zero_fraction']].rename(columns={'zero_fraction':'value'})
activity = activity.assign(scale='raw', statistic='zero_fraction', domain='annual')
curves = pd.concat([levels, changes, activity], ignore_index=True)
KEYS = ['domain', 'scale', 'statistic', 'age']
reference = curves.loc[curves.source.eq(EMPIRICAL), KEYS+['value', 'n']].rename(columns={'value':'empirical', 'n':'n_empirical'})
errors = curves.loc[curves.source.isin(MODELS)].rename(columns={'value':'simulated', 'n':'n_simulated'}).merge(reference, on=KEYS)
errors['difference'] = errors.simulated-errors.empirical
errors['squared_error'] = errors.difference**2
errors['absolute_error'] = errors.difference.abs()
prefix_end = np.where(errors.domain.eq('annual'), 5, 6)
errors['includes_empirical_prefix'] = errors.source.eq('ar6_6') & errors.age.le(prefix_end)
errors.to_csv(OUTPUT/'annual_and_increment_errors.csv', index=False)

In [5]:
def curve_score(frame):
    difference = frame.difference.to_numpy()
    empirical = frame.empirical.to_numpy()
    rmse = np.sqrt(np.mean(difference**2))
    scale = np.sqrt(np.mean(empirical**2))
    return {'n_years':len(frame), 'start_age':int(frame.age.min()), 'end_age':int(frame.age.max()),
            'mean_bias':difference.mean(), 'mae':np.mean(np.abs(difference)), 'rmse':rmse,
            'max_abs_error':np.max(np.abs(difference)), 'empirical_rms':scale,
            'nrmse':rmse/scale if scale>0 else np.nan,
            'includes_empirical_prefix':bool(frame.includes_empirical_prefix.any())}
score_rows = []
for window in ['post_initialization', 'common_generated']:
    start = np.ones(len(errors), dtype=int) if window=='post_initialization' else np.where(errors.domain.eq('annual'), COMMON_START, COMMON_START+1)
    selected = errors.loc[errors.age.ge(start)]
    for keys, frame in selected.groupby(['source', 'domain', 'scale', 'statistic'], sort=False):
        score_rows.append(dict(zip(['source', 'domain', 'scale', 'statistic'], keys), window=window, **curve_score(frame)))
curve_scores = pd.DataFrame(score_rows)
curve_scores.to_csv(OUTPUT/'curve_agreement.csv', index=False)
display(curve_scores.loc[curve_scores.domain.eq('annual') & curve_scores.scale.eq('raw')
                         & curve_scores.statistic.isin(['mean', 'median', 'variance'])])

,source,domain,scale,statistic,window,n_years,start_age,end_age,mean_bias,mae,rmse,max_abs_error,empirical_rms,nrmse,includes_empirical_prefix
0,arw4,annual,raw,mean,post_initialization,20,1,20,-2.930955e-01,4.115581e-01,5.017964e-01,8.633827e-01,7.434521,6.749546e-02,False
2,lognormal_grw,annual,raw,mean,post_initialization,20,1,20,2.842264e+03,2.842264e+03,4.988113e+03,1.416595e+04,7.434521,6.709394e+02,False
4,ar1_s,annual,raw,mean,post_initialization,20,1,20,1.357073e-01,4.024004e-01,4.616334e-01,8.719371e-01,7.434521,6.209323e-02,False
6,hurdle_ar1_s_p,annual,raw,mean,post_initialization,20,1,20,-8.729542e-01,8.729542e-01,9.209945e-01,1.284259e+00,7.434521,1.238808e-01,False
8,ar6_6,annual,raw,mean,post_initialization,20,1,20,-8.000986e-01,8.088582e-01,9.396094e-01,1.487297e+00,7.434521,1.263847e-01,True
10,se_hurdle_s,annual,raw,mean,post_initialization,20,1,20,-7.381988e-02,2.058269e-01,2.408783e-01,5.134010e-01,7.434521,3.239998e-02,False
12,arw4,annual,raw,median,post_initialization,20,1,20,2.564764e-01,3.307364e-01,4.172853e-01,9.476739e-01,5.667484,7.362797e-02,False
14,lognormal_grw,annual,raw,median,post_initialization,20,1,20,-1.675021e-01,9.106404e-01,1.062485e+00,2.086558e+00,5.667484,1.874703e-01,False
16,ar1_s,annual,raw,median,post_initialization,20,1,20,-1.558256e+00,1.558256e+00,1.615708e+00,2.336568e+00,5.667484,2.850838e-01,False
18,hurdle_ar1_s_p,annual,raw,median,post_initialization,20,1,20,-9.845098e-01,9.845098e-01,1.045330e+00,1.810670e+00,5.667484,1.844434e-01,False


In [6]:
# ranks compare distributions of relative positions, without pairing synthetic researchers to real ones
rank_empirical = ranks.loc[ranks.source.eq(EMPIRICAL), ['reference', 'age', 'rho']].rename(columns={'rho':'empirical'})
rank_errors = ranks.loc[ranks.source.isin(MODELS)].rename(columns={'rho':'simulated'}).merge(rank_empirical, on=['reference', 'age'])
rank_errors['difference'] = rank_errors.simulated-rank_errors.empirical
rank_errors['initial_rank_conditioned'] = rank_errors.reference.eq('age_0')
# years 11--20 make both lag-3 and lag-5 observations generated for AR(6,6)
rank_rows = []
for window, start in [('post_initialization', 1), ('common_late', COMMON_START+5)]:
    for (source, reference), frame in rank_errors.loc[rank_errors.age.ge(start)].groupby(['source', 'reference'], sort=False):
        difference = frame.difference.to_numpy()
        rank_rows.append({'source':source, 'reference':reference, 'window':window, 'n_years':len(frame),
                          'start_age':int(frame.age.min()), 'end_age':int(frame.age.max()),
                          'mean_bias':difference.mean(), 'rmse':np.sqrt(np.mean(difference**2)),
                          'max_abs_error':np.max(np.abs(difference)),
                          'initial_rank_conditioned':bool(frame.initial_rank_conditioned.any()),
                          'all_pairs_generated':bool(frame.both_generated.all()) and reference!='age_0'})
rank_scores = pd.DataFrame(rank_rows)
rank_errors.to_csv(OUTPUT/'rank_errors.csv', index=False)
rank_scores.to_csv(OUTPUT/'rank_agreement.csv', index=False)
display(rank_scores.loc[rank_scores.window.eq('common_late')])

,source,reference,window,n_years,start_age,end_age,mean_bias,rmse,max_abs_error,initial_rank_conditioned,all_pairs_generated
18,arw4,age_0,common_late,10,11,20,-0.223674,0.226145,0.272928,True,False
19,arw4,lag_3,common_late,10,11,20,-0.412352,0.414606,0.475324,False,True
20,arw4,lag_5,common_late,10,11,20,-0.451129,0.453604,0.505462,False,True
21,lognormal_grw,age_0,common_late,10,11,20,-0.053525,0.055432,0.086221,True,False
22,lognormal_grw,lag_3,common_late,10,11,20,0.197649,0.200707,0.263539,False,True
23,lognormal_grw,lag_5,common_late,10,11,20,0.177915,0.180233,0.226013,False,True
24,ar1_s,age_0,common_late,10,11,20,-0.234506,0.236766,0.274543,True,False
25,ar1_s,lag_3,common_late,10,11,20,-0.355766,0.358143,0.416293,False,True
26,ar1_s,lag_5,common_late,10,11,20,-0.416984,0.419343,0.482577,False,True
27,hurdle_ar1_s_p,age_0,common_late,10,11,20,-0.218594,0.220046,0.260448,True,False


In [7]:
# pooled-duration hazards; empirical risk weights define the supported durations
risk = hazards.loc[hazards.source.eq(EMPIRICAL) & hazards.n_at_risk.ge(MIN_REENTRY_RISK),
                   ['duration', 'n_at_risk', 'restart_probability']].rename(
    columns={'n_at_risk':'n_empirical_risk', 'restart_probability':'empirical'})
hazard_frames, hazard_rows = [], []
for source in MODELS:
    fitted = hazards.loc[hazards.source.eq(source), ['duration', 'n_at_risk', 'restart_probability']].rename(
        columns={'n_at_risk':'n_simulated_risk', 'restart_probability':'simulated'})
    frame = risk.merge(fitted, on='duration', how='left').assign(source=source)
    frame['difference'] = frame.simulated-frame.empirical
    weights = frame.n_empirical_risk/frame.n_empirical_risk.sum()
    # undefined simulated hazards remain missing; no silent drop of unsupported durations
    complete = len(frame)>0 and frame.simulated.notna().all()
    weighted_rmse = np.sqrt(np.sum(weights*frame.difference**2)) if complete else np.nan
    hazard_rows.append({'source':source, 'min_empirical_risk':MIN_REENTRY_RISK, 'n_durations':len(frame),
                        'n_matched':int(frame.simulated.notna().sum()), 'risk_weighted_rmse':weighted_rmse,
                        'simulated_duration_coverage':float(weights[frame.simulated.notna()].sum()),
                        'contains_prefix_transitions':False, 'pooling':'all generated transitions'})
    hazard_frames.append(frame)
pd.concat(hazard_frames, ignore_index=True).to_csv(OUTPUT/'reentry_errors.csv', index=False)
hazard_scores = pd.DataFrame(hazard_rows)
hazard_scores.to_csv(OUTPUT/'reentry_agreement.csv', index=False)
display(hazard_scores)

,source,min_empirical_risk,n_durations,n_matched,risk_weighted_rmse,simulated_duration_coverage,contains_prefix_transitions,pooling
0,arw4,20,5,0,NaN,0.0,False,all generated transitions
1,lognormal_grw,20,5,5,0.148733,1.0,False,all generated transitions
2,ar1_s,20,5,5,0.213554,1.0,False,all generated transitions
3,hurdle_ar1_s_p,20,5,5,0.075087,1.0,False,all generated transitions
4,ar6_6,20,5,5,0.075844,1.0,False,all generated transitions
5,se_hurdle_s,20,5,5,0.020266,1.0,False,all generated transitions


In [8]:
# ready-to-use statements; percentages are RMSE / empirical-curve RMS, not percentage bias
statement_rows = []
for (source, window), frame in curve_scores.loc[curve_scores.domain.eq('annual') & curve_scores.scale.eq('raw')].groupby(['source', 'window'], sort=False):
    values = frame.set_index('statistic')
    first, last = int(values.start_age.iloc[0]), int(values.end_age.iloc[0])
    percentages = [100*values.loc[name, 'nrmse'] for name in ['mean', 'median', 'variance']]
    text = (f'Over career years {first}--{last}, {LABELS[source]} had normalized RMSEs of '
            f'{percentages[0]:.3g}%, {percentages[1]:.3g}%, and {percentages[2]:.3g}% '
            'for annual mean, median, and variance, respectively; each error is normalized '
            'by the root mean square of its empirical curve.')
    statement_rows.append({'source':source, 'window':window, 'option':'annual population curves', 'text':text,
                           'includes_empirical_prefix':bool(frame.includes_empirical_prefix.any())})
for row in rank_scores.loc[rank_scores.window.eq('common_late') & rank_scores.reference.eq('lag_5')].itertuples():
    statement_rows.append({'source':row.source, 'window':row.window, 'option':'rank persistence',
                           'text':f'Over career years {row.start_age}--{row.end_age}, {LABELS[row.source]} had lag-five annual-rank correlation RMSE {row.rmse:.4f}.',
                           'includes_empirical_prefix':not row.all_pairs_generated})
statements = pd.DataFrame(statement_rows)
statements.to_csv(OUTPUT/'discussion_options.csv', index=False)
display(statements.loc[statements.window.eq('common_generated'), ['source', 'text']])

,source,text
6,arw4,"Over career years 6--20, ARW4 had normalized R..."
7,lognormal_grw,"Over career years 6--20, GRW had normalized RM..."
8,ar1_s,"Over career years 6--20, AR(1) had normalized ..."
9,hurdle_ar1_s_p,"Over career years 6--20, Hurdle AR(1) had norm..."
10,ar6_6,"Over career years 6--20, AR(6,6) had normalize..."
11,se_hurdle_s,"Over career years 6--20, Self-exciting hurdle ..."


In [9]:
definitions = pd.DataFrame([
    {'metric':'annual mean/median/variance curves', 'interpretation':'evolution of the population distribution; does not establish individual path agreement'},
    {'metric':'increment mean/median/variance curves', 'interpretation':'year-to-year changes; destination age labels and fully generated endpoints'},
    {'metric':'NRMSE', 'interpretation':'RMSE divided by RMS of empirical curve over the same years; undefined when empirical RMS is zero'},
    {'metric':'common_generated annual', 'interpretation':f'career years {COMMON_START}--{Y}; generated values for every model'},
    {'metric':'common_generated increment', 'interpretation':f'destination years {COMMON_START+1}--{Y}; both endpoints generated for every model'},
    {'metric':'common_late rank', 'interpretation':f'years {COMMON_START+5}--{Y}; lag-3/5 pairs generated; initial-rank comparisons remain initial-conditioned'},
    {'metric':'reentry agreement', 'interpretation':f'pooled generated transitions, empirical complete-cohort risk weights, at least {MIN_REENTRY_RISK} empirical cases per duration; not age-standardized'},
    {'metric':'fit comparison', 'interpretation':'SE fits complete careers; others use available observations; in-sample checks without uncertainty, not a controlled ablation'}
])
definitions.to_csv(OUTPUT/'metric_definitions.csv', index=False)
display(definitions)

,metric,interpretation
0,annual mean/median/variance curves,evolution of the population distribution; does...
1,increment mean/median/variance curves,year-to-year changes; destination age labels a...
2,NRMSE,RMSE divided by RMS of empirical curve over th...
3,common_generated annual,career years 6--20; generated values for every...
4,common_generated increment,destination years 7--20; both endpoints genera...
5,common_late rank,years 11--20; lag-3/5 pairs generated; initial...
6,reentry agreement,"pooled generated transitions, empirical comple..."
7,fit comparison,SE fits complete careers; others use available...
